# 04 · Eleição do vencedor e promoção controlada

Passos **7, 8 e 9 do fluxo único**:

1. **Eleição automática** do vencedor do ciclo pela média da métrica principal na validação cruzada (empate técnico → menor desvio-padrão);
2. **Checagens de integridade** — mesmo `cycle_id`/`data_version` em votação, treinos e decisão; mesmas features nos dois candidatos; hash do modelo vigente conferido. Qualquer falha **bloqueia** a promoção e interrompe o fluxo;
3. **Desafiante × vigente** no holdout de **validação** (mesmos registros para os dois → protocolo compatível; o vigente usa o próprio pré-processamento e as próprias features);
4. **Regra de promoção** (definida a priori no `config.yaml`): promover só se superar o vigente em pelo menos o limiar de melhoria **e** atender aos pisos de qualidade; caso contrário, o vigente é mantido;
5. Registro em `models/modelo_vigente.json`, em `models/historico_decisoes.csv`, no MLflow (incluindo Model Registry com alias `vigente`). Versões anteriores são preservadas em `models/versoes/`.

In [1]:
import sys
from pathlib import Path

# Caminhos consistentes: localiza a raiz do projeto (pasta com config/config.yaml)
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
from src.config import preparar_notebook

cfg = preparar_notebook()  # muda o diretório de trabalho para a raiz do projeto

import json

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 80, "figure.figsize": (9, 4)})

from mlflow.tracking import MlflowClient

from src import ciclo, ingestao, modelagem, promocao, rastreamento
from src import preprocessamento as pp
from src.config import ler_json, salvar_json

ctx = ciclo.carregar_ciclo(exigir=["01_etl", "02_votacao", "03_treino"])
EVID = ciclo.dir_evidencias(ctx)
metrica = cfg["avaliacao"]["metrica_principal"]
classes = cfg["dados"]["classes"]
print(f"cycle_id = {ctx['cycle_id']} | data_version = {ctx['data_version']}")

cycle_id = c0009-20261127T185643Z | data_version = dv009-d59b5486


## 1. Eleição automática do vencedor do ciclo

In [2]:
cv = pd.read_csv(EVID / "treino_cv_folds.csv")
resumo = promocao.resumir_cv(cv, metrica).sort_values("papel").reset_index(drop=True)
display(resumo.round(4))
eleicao = promocao.eleger_vencedor(resumo, cfg)
print(f"Vencedor do ciclo: {eleicao['vencedor']} ({eleicao['algoritmo_vencedor']}) - {eleicao['motivo']}")

,papel,algoritmo,media,desvio,minimo,maximo
0,modelo_a,random_forest,0.7350,0.0066,0.7256,0.7430
1,modelo_b,xgboost,0.7599,0.0042,0.7560,0.7666


Vencedor do ciclo: modelo_b (xgboost) - maior média de f1_macro na validação cruzada (Δ=0.0249)


## 2. Checagens de integridade (falha = promoção bloqueada)

In [3]:
treino_info = ler_json(EVID / "treino_candidatos.json")
votacao = ler_json(EVID / "votacao_features.json")
runs = rastreamento.runs_do_ciclo(cfg, ctx["cycle_id"])
runs_treino = runs[runs["tags.etapa"] == "03_treino"]
registro = ingestao.carregar_registro(cfg)
cands = treino_info["candidatos"]
checagens = [
    {"regra": "integridade_cycle_id", "ok": votacao["cycle_id"] == treino_info["cycle_id"] == ctx["cycle_id"],
     "detalhe": "votação, treinos e decisão com o mesmo cycle_id"},
    {"regra": "integridade_data_version", "ok": votacao["data_version"] == treino_info["data_version"] == ctx["data_version"] == registro.get("data_version_atual"),
     "detalhe": f"data_version {ctx['data_version']} em todas as etapas e no registro de lotes"},
    {"regra": "integridade_dois_candidatos", "ok": len(runs_treino) == 2 and set(runs_treino["tags.algoritmo"]) == {c["algoritmo"] for c in cands.values()},
     "detalhe": f"runs de treino no MLflow para o ciclo: {sorted(runs_treino['tags.algoritmo'])}"},
    {"regra": "integridade_mesmas_features", "ok": all(c["features"] == votacao["eleitas"] for c in cands.values()),
     "detalhe": "modelos A e B usam exatamente as features eleitas no ciclo"},
]
display(pd.DataFrame(checagens))

,regra,ok,detalhe
0,integridade_cycle_id,True,"votação, treinos e decisão com o mesmo cycle_id"
1,integridade_data_version,True,data_version dv009-d59b5486 em todas as etapas...
2,integridade_dois_candidatos,True,runs de treino no MLflow para o ciclo: ['rando...
3,integridade_mesmas_features,True,modelos A e B usam exatamente as features elei...


## 3. Desafiante × vigente no holdout de validação

In [4]:
base = pd.read_parquet(Path(cfg["dados"]["dir_processed"]) / "base_features.parquet")
val = base[base["particao"] == "validacao"]
X_val, y_val = val[pp.COLUNAS_FEATURES], modelagem.codificar_alvo(val["type"], classes)

candidato = cands[eleicao["vencedor"]]
desafiante = promocao.carregar_modelo(candidato["arquivo_bundle"])
m_desafiante = modelagem.calcular_metricas(y_val, desafiante.prever_proba_features(X_val), classes)
perdedor = promocao.carregar_modelo(cands[eleicao["perdedor"]]["arquivo_bundle"])
m_perdedor = modelagem.calcular_metricas(y_val, perdedor.prever_proba_features(X_val), classes)

try:
    manifesto, vigente = promocao.carregar_vigente()
    checagens.append({"regra": "integridade_vigente", "ok": True,
                      "detalhe": "sem vigente" if vigente is None else f"vigente {manifesto['vigente']['versao']} carregado e hash conferido"})
except Exception as erro:  # arquivo ausente/adulterado: bloqueia a promoção
    manifesto, vigente = None, None
    checagens.append({"regra": "integridade_vigente", "ok": False, "detalhe": str(erro)})
m_vigente = None if vigente is None else modelagem.calcular_metricas(y_val, vigente.prever_proba_features(X_val), classes)

comparacao = pd.DataFrame({f"desafiante ({eleicao['algoritmo_vencedor']})": m_desafiante,
                           f"perdedor do ciclo ({eleicao['algoritmo_perdedor']}) - informativo": m_perdedor,
                           **({f"vigente ({manifesto['vigente']['versao']}, {manifesto['vigente']['algoritmo']})": m_vigente} if m_vigente else {})})
display(comparacao.round(4))

,desafiante (xgboost),perdedor do ciclo (random_forest) - informativo,"vigente (v0003, xgboost)"
f1_macro,0.7594,0.7317,0.7705
accuracy,0.7488,0.7080,0.7648
balanced_accuracy,0.8103,0.7797,0.8165
precision_macro,0.7448,0.7341,0.7561
recall_macro,0.8103,0.7797,0.8165
log_loss,0.5371,0.6645,0.5141
f1_benign,0.8004,0.7627,0.8156
recall_benign,0.7075,0.6563,0.7305
f1_defacement,0.8190,0.7845,0.8292
recall_defacement,0.8908,0.8332,0.8914


## 4. Regra de promoção

In [5]:
decisao = promocao.decidir_promocao(m_desafiante, m_vigente, cfg, checagens)
display(pd.DataFrame(decisao["checagens"]))
display(Markdown(f"### Decisão automática: **{decisao['decisao'].upper()}**\n\n{decisao['motivo']}"))

,regra,ok,detalhe
0,integridade_cycle_id,True,"votação, treinos e decisão com o mesmo cycle_id"
1,integridade_data_version,True,data_version dv009-d59b5486 em todas as etapas...
2,integridade_dois_candidatos,True,runs de treino no MLflow para o ciclo: ['rando...
3,integridade_mesmas_features,True,modelos A e B usam exatamente as features elei...
4,integridade_vigente,True,vigente v0003 carregado e hash conferido
5,metrica_finita,True,f1_macro do desafiante = 0.7593960229631429
6,piso_metrica_principal,True,f1_macro=0.7594 vs piso 0.7
7,recall_minimo_por_classe,True,menor recall por classe=0.7075 vs mínimo 0.5
8,supera_vigente,False,Δ=-0.0111 (desafiante 0.7594 x vigente 0.7705)...


### Decisão automática: **MANTER**

supera_vigente: Δ=-0.0111 (desafiante 0.7594 x vigente 0.7705); limiar 0.002

## 5. Registro da decisão (MLflow, Model Registry e manifesto)

In [6]:
nome_registro = cfg["mlflow"]["modelo_registrado"]
with rastreamento.run_etapa(ctx, cfg, "04_promocao", "04_promocao", tags={"decisao": decisao["decisao"], "vencedor_ciclo": eleicao["vencedor"]}) as run:
    mlflow.log_params({"regra_limiar_melhoria": cfg["promocao"]["limiar_melhoria"], "regra_metrica_minima": cfg["promocao"]["metrica_minima"],
                       "regra_recall_minimo_por_classe": cfg["promocao"]["recall_minimo_por_classe"], "metrica_principal": metrica,
                       "direcao": cfg["avaliacao"]["direcao"], "conjunto_comparacao": cfg["promocao"]["conjunto_comparacao"],
                       "vencedor_ciclo": eleicao["vencedor"], "algoritmo_vencedor": eleicao["algoritmo_vencedor"],
                       "run_id_desafiante": candidato["run_id"], "decisao": decisao["decisao"]})
    mlflow.log_metrics({"cv_vencedor": eleicao["media_vencedor"], "cv_perdedor": eleicao["media_perdedor"],
                        f"validacao_{metrica}_desafiante": m_desafiante[metrica]}
                       | ({f"validacao_{metrica}_vigente": m_vigente[metrica], "delta": decisao["delta"]} if m_vigente else {}))
    if decisao["decisao"] == "bloqueado":
        mlflow.log_dict(decisao, "decisao_promocao.json")
        raise promocao.PromocaoBloqueada(decisao["motivo"])
    if decisao["decisao"] == "promover":
        versao_mlflow = mlflow.register_model(candidato["model_uri"], nome_registro, tags={"cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"]})
        MlflowClient().set_registered_model_alias(nome_registro, cfg["mlflow"]["alias_vigente"], versao_mlflow.version)
        candidato["modelo_registrado"] = {"nome": nome_registro, "versao": versao_mlflow.version, "alias": cfg["mlflow"]["alias_vigente"]}
    candidato["metricas_validacao"] = m_desafiante
    decisao["run_id_promocao"] = run.info.run_id
    manifesto = promocao.registrar_decisao(ctx, cfg, decisao, eleicao, candidato, desafiante)
    resultado = {"cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "eleicao": eleicao, "decisao": decisao,
                 "metricas_validacao": {"desafiante": m_desafiante, "perdedor": m_perdedor, "vigente": m_vigente}}
    salvar_json(resultado, EVID / "decisao_promocao.json")
    comparacao.to_csv(EVID / "promocao_comparacao_validacao.csv")
    mlflow.log_dict(resultado, "decisao_promocao.json")
    mlflow.log_artifact("models/modelo_vigente.json")
    mlflow.log_artifact("models/historico_decisoes.csv")
    run_id = run.info.run_id

rastreamento.atualizar_run_pai(ctx, tags={"decisao": decisao["decisao"], "vencedor_ciclo": eleicao["algoritmo_vencedor"],
                                          "versao_vigente": (manifesto.get("vigente") or {}).get("versao", "-")})
ctx = ciclo.registrar_etapa(ctx, "04_promocao", run_id=run_id, decisao=decisao["decisao"],
                            versao_vigente=(manifesto.get("vigente") or {}).get("versao"))
print(json.dumps({k: manifesto[k] for k in ("vigente", "ultima_decisao")}, indent=2, ensure_ascii=False, default=str)[:3000])

🏃 View run c0009-20261127T185643Z/04_promocao at: http://127.0.0.1:5000/#/experiments/1/runs/61277da6e3ab40db8e1311e822d7522b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
{
  "vigente": {
    "versao": "v0003",
    "cycle_id": "c0006-20261124T200921Z",
    "data_version": "dv006-f2efc620",
    "algoritmo": "xgboost",
    "papel_no_ciclo": "modelo_b",
    "features": [
      "kw_cms",
      "num_params",
      "path_length",
      "num_path_tokens",
      "num_letters",
      "num_ampersand",
      "tld_com",
      "num_slash",
      "query_length",
      "path_depth",
      "has_query",
      "num_equal",
      "digits_in_host",
      "ratio_special",
      "ratio_letters",
      "ratio_digits",
      "url_length",
      "num_digits",
      "has_exec_extension",
      "has_ip",
      "entropy",
      "num_special",
      "kw_malware",
      "num_hyphen",
      "num_dots",
      "host_length",
      "num_question",
      "longest_token",
      "num_suspicious_words",
    